In [1]:
import tkinter as tk
from tkinter import filedialog, messagebox
from tkcalendar import DateEntry
import pandas as pd
import gspread
from google.oauth2.service_account import Credentials
import warnings

warnings.filterwarnings("ignore")

# ---------------- CONFIGURATION ----------------

SCOPES = [
    "https://www.googleapis.com/auth/spreadsheets",
    "https://www.googleapis.com/auth/drive.readonly"
]

DEFAULT_MTD_LINK = "https://docs.google.com/spreadsheets/d/1-n_dg-d1wAgNtZxk3hOyPQCS2SGY8nYFnITNFbyQ-lw/edit?gid=1852609864#gid=1852609864"

CURRENT_SPEND_SHEET = "https://docs.google.com/spreadsheets/d/1fh_r5--XHnWjM5ZTOQLBDu2eUJkDdyFh2bW4y8-aA6U/edit?gid=473136687#gid=473136687"
PREV_SPEND_SHEET = "https://docs.google.com/spreadsheets/d/1Z5wuZEQ_cJUL93x1y6mSCrV43-vYsnFOZQ8upda5cj0/edit?gid=1040806638#gid=1040806638"

# ---------------- DUREX CONSTANTS ----------------

DUREX_UTM_LIST = ["bik", "cem", "crm", "crn", "kwikchat", "kwikengage", "sms", "tcway_crm", "wa", "whatsapp", "whatsapp_order_shipped", "whatsApp", "social_collection_Whatsapp"]
DUREX_UTM_FOR_CODES = ["affilate","affiliate", "aff_paisawapas","netmax","D2C_Ninja", "affiliates", "cashkro", "direct", "grabon", "partnership", "smile_rewards", "social_brand", "manifest","Affilaite", "Affiliate", "affiliates", "Affiliates", "Cashkaro", "social", "Affilaite", "affiliate","Partnership", "affiliates", "an", "Cashkaro", "chatgpt.com", "direct", "discoveringbrands", "durex_social", "grabon", "hamburger", "manifest", "NA", "PaisaWapas.com", "Partnership", "perplexity", "pwa", "scan_qr", "smile_rewards", "social_brand", "th", "pokkt","bik", "cem", "crm", "crn", "kwikchat", "kwikengage", "sms", "tcway_crm", "wa", "whatsapp", "whatsapp_order_shipped", "whatsApp", "social_collection_Whatsapp"]
DUREX_COUPON_LIST = ["XQZ745","VXT613","ZRM452","BQD987","HVD721","WPF748","CKM591","DXT374","GZR865","NXQ427","TY32","WS58","VD72","KP93","TYL309","MWS582","YLN236","RKP934","LQB658","QZ74","XT61","RM45","QD98","PF74","KM59","XT37","ZR86","XQ42","CART10","CART15","GL58","CART05","PAYDAY15","PAYDAY20","Buy3@699","NEW20"]

# ---------------- HELPERS ----------------

def clean_headers(df):
    df.columns = df.columns.astype(str).str.strip().str.lower()
    return df


# ---------------- CSV PROCESSING ----------------

def process_durex_csv(
    csv_path,
    start_date,
    end_date,
    utm_filter,
    coupon_filter=None
):

    if not csv_path:
        return 0

    try:

        df = pd.read_csv(csv_path)

        df = clean_headers(df)

        df["created at"] = pd.to_datetime(
            df["created at"],
            errors='coerce',
            dayfirst=True
        ).dt.date

        df = df.dropna(subset=["created at"])

        df = df[
            (df["created at"] >= start_date.date()) &
            (df["created at"] <= end_date.date())
        ]

        utm_col = [c for c in df.columns if "utm" in c][0]

        df["temp_utm"] = (
            df[utm_col]
            .astype(str)
            .str.lower()
            .str.strip()
        )

        df = df[
            df["temp_utm"].isin(
                [x.lower() for x in utm_filter]
            )
        ]

        if coupon_filter:

            coupon_col = [
                c for c in df.columns
                if "coupon" in c
            ][0]

            df["temp_coupon"] = (
                df[coupon_col]
                .astype(str)
                .str.upper()
                .str.strip()
            )

            df = df[
                df["temp_coupon"].isin(
                    [x.upper() for x in coupon_filter]
                )
            ]

        total = pd.to_numeric(
            df["grand total"],
            errors='coerce'
        ).sum()

        return round(total)

    except Exception as e:
        print("CSV ERROR:", e)
        return 0


# ---------------- SPENDS FUNCTION ----------------

def get_spends_value(
    sheet_url,
    day_input,
    header_row,
    spend_column
):

    try:

        creds = Credentials.from_service_account_file(
            "service_account.json",
            scopes=SCOPES
        )

        client = gspread.authorize(creds)

        sheet = client.open_by_url(sheet_url).worksheet(
            "Daily Spends & Revenue Tracker"
        )

        data = sheet.get_all_values()

        df = pd.DataFrame(
            data[header_row:],
            columns=data[header_row - 1]
        )

        df.columns = (
            df.columns
            .str.strip()
            .str.lower()
        )

        # FIND DATE COLUMN
        date_col = [
            c for c in df.columns
            if "date" in c
        ][0]

        # EXTRACT DAY
        df["day"] = (
            df[date_col]
            .astype(str)
            .str.extract(r"(\d+)")
            .astype(float)
        )

        # FILTER TILL SELECTED DAY
        df = df[
            df["day"].notna() &
            (df["day"] <= day_input)
        ]

        # COLUMN LETTER TO INDEX
        spend_index = (
            ord(spend_column.upper()) - ord('A')
        )

        spend_data = pd.to_numeric(
            df.iloc[:, spend_index]
            .astype(str)
            .str.replace(r'[₹,]', '', regex=True)
            .str.strip(),
            errors='coerce'
        )

        return round(float(spend_data.sum()))

    except Exception as e:
        print("SPEND ERROR:", e)
        return 0


# ---------------- FIND BLOCK ----------------

def find_block(sheet, brand, end_date):

    target = f"{brand} - {end_date.strftime('%b')}".lower()

    data = sheet.get_all_values()

    for i, row in enumerate(data):

        if any(
            str(c).strip().lower() == target
            for c in row
        ):
            return i + 1

    raise Exception(
        f"Block '{target}' not found."
    )


# ---------------- MAIN SCRIPT ----------------

def run_durex_script():

    try:

        status_var.set("Running...")
        root.update()

        creds = Credentials.from_service_account_file(
            "service_account.json",
            scopes=SCOPES
        )

        client = gspread.authorize(creds)

        start = pd.to_datetime(start_cal.get_date())
        end = pd.to_datetime(end_cal.get_date())

        # ---------------- CURRENT REVENUE ----------------

        utm_cur = process_durex_csv(
            current_file.get(),
            start,
            end,
            DUREX_UTM_LIST
        )

        codes_cur = process_durex_csv(
            current_file.get(),
            start,
            end,
            DUREX_UTM_FOR_CODES,
            DUREX_COUPON_LIST
        )

        h_cur = round(
            float(headless_current.get() or 0)
        )

        total_cur = round(
            h_cur + utm_cur + codes_cur
        )

        # ---------------- PREVIOUS REVENUE ----------------

        p_start = start - pd.DateOffset(months=1)
        p_end = end - pd.DateOffset(months=1)

        utm_prev = process_durex_csv(
            prev_file.get(),
            p_start,
            p_end,
            DUREX_UTM_LIST
        )

        codes_prev = process_durex_csv(
            prev_file.get(),
            p_start,
            p_end,
            DUREX_UTM_FOR_CODES,
            DUREX_COUPON_LIST
        )

        h_prev = round(
            float(headless_previous.get() or 0)
        )

        total_prev = round(
            h_prev + utm_prev + codes_prev
        )

        # ---------------- SPENDS ----------------

        spend_cur = get_spends_value(
            CURRENT_SPEND_SHEET,
            end.day,
            int(current_header_row_var.get()),
            current_spend_column_var.get()
        )

        spend_prev = get_spends_value(
            PREV_SPEND_SHEET,
            p_end.day,
            int(previous_header_row_var.get()),
            previous_spend_column_var.get()
        )

        # ---------------- GOOGLE SHEET ----------------

        sheet = client.open_by_url(
            DEFAULT_MTD_LINK
        ).worksheet("DUREX")

        row_idx = find_block(
            sheet,
            "Durex",
            end
        )

        h_row = row_idx + 4
        hl_row = row_idx + 5
        u_row = row_idx + 6
        c_row = row_idx + 7
        t_row = row_idx + 8
        s_row = row_idx + 9
        r_row = row_idx + 10

        # ---------------- HISTORY SHIFT ----------------

        for col in range(20, 6, -1):

            curr_c = chr(64 + col)
            prev_c = chr(64 + col - 1)

            vals = sheet.get(
                f"{prev_c}{h_row}:{prev_c}{s_row}"
            )

            if vals and any(v[0] for v in vals if v):

                sheet.update(
                    f"{curr_c}{h_row}",
                    vals
                )

        # ---------------- MOVE CURRENT TO HISTORY ----------------

        old_h = sheet.acell(f"B{h_row}").value

        old_v = sheet.get(
            f"B{hl_row}:B{s_row}"
        )

        if old_h:

            sheet.update(
                f"F{h_row}",
                [[old_h]]
            )

            sheet.update(
                f"F{hl_row}",
                old_v
            )

        # ---------------- UPDATES ----------------

        days_in_mo = pd.Timestamp(
            end
        ).days_in_month

        updates = [

            {'range': f"B{h_row}",
             'values': [[f"Till {end.day}th"]]},

            {'range': f"B{hl_row}",
             'values': [[h_cur]]},

            {'range': f"B{u_row}",
             'values': [[utm_cur]]},

            {'range': f"B{c_row}",
             'values': [[codes_cur]]},

            {'range': f"B{t_row}",
             'values': [[total_cur]]},

            {'range': f"B{s_row}",
             'values': [[spend_cur]]},

            {'range': f"E{hl_row}",
             'values': [[h_prev]]},

            {'range': f"E{u_row}",
             'values': [[utm_prev]]},

            {'range': f"E{c_row}",
             'values': [[codes_prev]]},

            {'range': f"E{t_row}",
             'values': [[total_prev]]},

            {'range': f"E{s_row}",
             'values': [[spend_prev]]},

            # TARGETS

            {'range': f"C{t_row}",
             'values': [[
                 f"=(C{row_idx+2}/{days_in_mo})*{end.day}"
             ]]},

            {'range': f"D{t_row}",
             'values': [[
                 f"=IFERROR(B{t_row}/C{t_row},0)"
             ]]},

            {'range': f"C{s_row}",
             'values': [[
                 f"=(A{row_idx+2}/{days_in_mo})*{end.day}"
             ]]},

            {'range': f"D{s_row}",
             'values': [[
                 f"=IFERROR(B{s_row}/C{s_row},0)"
             ]]}
        ]

        # ---------------- ROAS ----------------

        for col_l in ['B', 'E']:

            updates.append({

                'range': f"{col_l}{r_row}",

                'values': [[
                    f"=IFERROR({col_l}{t_row}/{col_l}{s_row},0)"
                ]]
            })

        # ---------------- FINAL UPDATE ----------------

        sheet.batch_update(
            updates,
            value_input_option="USER_ENTERED"
        )

        status_var.set("Done ✅")

        messagebox.showinfo(
            "Success",
            "Process completed successfully!"
        )

    except Exception as e:

        status_var.set("Error ❌")

        messagebox.showerror(
            "Error",
            str(e)
        )


# ---------------- TKINTER UI ----------------

root = tk.Tk()

root.title("Durex MTD Revenue Automation")

root.geometry("850x700")

status_var = tk.StringVar(value="Idle")

current_file = tk.StringVar()
prev_file = tk.StringVar()

headless_current = tk.StringVar(value="0")
headless_previous = tk.StringVar(value="0")

# CURRENT SHEET SETTINGS

current_header_row_var = tk.StringVar(value="3")
current_spend_column_var = tk.StringVar(value="T")

# PREVIOUS SHEET SETTINGS

previous_header_row_var = tk.StringVar(value="3")
previous_spend_column_var = tk.StringVar(value="T")

# ---------------- TITLE ----------------

tk.Label(
    root,
    text="DUREX REVENUE AUTOMATION",
    font=("Arial", 14, "bold")
).pack(pady=10)

# ---------------- CSV SECTION ----------------

f_csv = tk.LabelFrame(
    root,
    text="CSV Data Selection",
    padx=10,
    pady=10
)

f_csv.pack(fill="x", padx=20, pady=5)

tk.Button(
    f_csv,
    text="Current CSV",
    width=15,
    command=lambda:
    current_file.set(
        filedialog.askopenfilename()
    )
).grid(row=0, column=0)

tk.Label(
    f_csv,
    textvariable=current_file,
    fg="blue",
    wraplength=550
).grid(row=0, column=1, sticky="w", padx=10)

tk.Button(
    f_csv,
    text="Previous CSV",
    width=15,
    command=lambda:
    prev_file.set(
        filedialog.askopenfilename()
    )
).grid(row=1, column=0, pady=5)

tk.Label(
    f_csv,
    textvariable=prev_file,
    fg="blue",
    wraplength=550
).grid(row=1, column=1, sticky="w", padx=10)

# ---------------- HEADLESS ----------------

f_head = tk.Frame(root)

f_head.pack(pady=10)

tk.Label(
    f_head,
    text="Headless (Cur):"
).grid(row=0, column=0)

tk.Entry(
    f_head,
    textvariable=headless_current,
    width=12
).grid(row=0, column=1, padx=10)

tk.Label(
    f_head,
    text="Headless (Prev):"
).grid(row=0, column=2)

tk.Entry(
    f_head,
    textvariable=headless_previous,
    width=12
).grid(row=0, column=3, padx=10)

# ---------------- SPEND SETTINGS ----------------

f_spend = tk.LabelFrame(
    root,
    text="Spend Sheet Settings",
    padx=10,
    pady=10
)

f_spend.pack(fill="x", padx=20, pady=5)

# CURRENT SETTINGS

tk.Label(
    f_spend,
    text="Current Header Row:"
).grid(row=0, column=0, padx=5, pady=5)

tk.Entry(
    f_spend,
    textvariable=current_header_row_var,
    width=10
).grid(row=0, column=1, padx=5)

tk.Label(
    f_spend,
    text="Current Spend Column:"
).grid(row=0, column=2, padx=5)

tk.Entry(
    f_spend,
    textvariable=current_spend_column_var,
    width=10
).grid(row=0, column=3, padx=5)

# PREVIOUS SETTINGS

tk.Label(
    f_spend,
    text="Previous Header Row:"
).grid(row=1, column=0, padx=5, pady=5)

tk.Entry(
    f_spend,
    textvariable=previous_header_row_var,
    width=10
).grid(row=1, column=1, padx=5)

tk.Label(
    f_spend,
    text="Previous Spend Column:"
).grid(row=1, column=2, padx=5)

tk.Entry(
    f_spend,
    textvariable=previous_spend_column_var,
    width=10
).grid(row=1, column=3, padx=5)

# ---------------- DATES ----------------

f_date = tk.Frame(root)

f_date.pack(pady=10)

tk.Label(
    f_date,
    text="Start:"
).pack(side="left")

start_cal = DateEntry(f_date)

start_cal.pack(side="left", padx=5)

tk.Label(
    f_date,
    text="End:"
).pack(side="left", padx=5)

end_cal = DateEntry(f_date)

end_cal.pack(side="left")

# ---------------- BUTTON ----------------

tk.Button(
    root,
    text="🚀 START PROCESS",
    command=run_durex_script,
    bg="#28a745",
    fg="white",
    font=("Arial", 12, "bold"),
    height=2,
    width=30
).pack(pady=20)

# ---------------- STATUS ----------------

tk.Label(
    root,
    textvariable=status_var,
    font=("Arial", 10, "italic")
).pack()

root.mainloop()
